# Single-task FinBERT baselines on Kaggle (steps 1a-1c)

**Before running**, in the right-hand panel *Settings*:
1. *Accelerator*: **GPU T4 x2** (the code uses one GPU; T4 is the safe choice).
2. *Internet*: **on** (needed for GitHub and Hugging Face).
3. *Add-ons > Secrets*: add a secret named **`HF_TOKEN`** with a Hugging Face read token from the account that accepted the FiQA access form, and tick it for this notebook.

For the full run (about 1-2 h), use **Save Version > Save & Run All (Commit)**: it keeps running after you close the tab (12 h limit) and everything in `/kaggle/working` is kept as the notebook's output.

In [ ]:
%cd /kaggle/working
!git clone -q https://github.com/ptrnghieu/SEntFiN
%cd SEntFiN
!git checkout -q claude/adoring-fermat-r9ta0k && git log --oneline -1

In [ ]:
# torch is preinstalled on Kaggle; this adds the data-pipeline deps and pytorch-crf
!pip install -q -r requirements.txt pytorch-crf
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')  # read by huggingface_hub
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

In [ ]:
# Rebuild data/processed (~15 s): downloads FiQA (gated) and FinEntity; LLM labels are committed in the repo
!python -m absa_data build --require-fiqa

## Quick check first (about 2 min)
One short run per script before committing the full matrix.

In [ ]:
%%bash
set -euo pipefail  # a failing step stops the notebook here
python -m absa_models.target_extraction --train-sources finentity --crf --epochs 1 --out runs/check/extraction
python -m absa_models.pair_classify --task sentiment --train-sources finentity --eval-sources finentity --epochs 1 --out runs/check/sentiment
head -30 runs/check/sentiment/metrics.md

## Full matrix
Target extraction (3 runs), sentiment (5 runs), aspect (2 runs); results collected in `runs/SUMMARY.md`.

In [ ]:
%%bash
set -euo pipefail
rm -rf runs/check
bash scripts/run_baselines.sh 2>&1 | grep -v -i warning

In [ ]:
from IPython.display import Markdown
Markdown(open('runs/SUMMARY.md').read())

In [ ]:
# Optional: drop the checkpoints (~440 MB each) so the saved output stays small; metrics and predictions stay.
# !find runs -name best.pt -delete
!du -sh runs